# OghmaNano `01_hello_oled` — ITO/Al 半平面 TMM 对齐

Oghma v8.1 项目 [`01_hello_oled`](../../../database/og/oghma_projects/oled/01_hello_oled) 的 **`optical_output/`** 与 simulation TMM 逐 case 对比。

**TMM 上下半平面为 ITO / Al**；`light_illuminate_from=y0` → **ITO | ITO→Al | Al**，`TMM_get_r_t_power_s` / `TMM_get_r_t_power_p`。

**Phase 1 门控**：R(λ)/T(λ) 对齐通过后，才进入 photons / absorption / snapshots（Phase 2）。

**膜系**（y=0 @ ITO，310 nm）：ITO(∞) | ITO(150) / NPD(40) / Alq3(30) / TPBi(30) / LiF(10) / Al(50) | Al(∞)。半无限外场用 depth=0 的 ITO_pad/Al_pad 表示；有限 ITO/Al 接触层在 epitaxy 内（非 FDTD 式 finite air pad）。

**坐标**：epitaxy 从 `z=0` @ Oghma `y=0`（ITO 接触）起至 `z=310` @ Al 顶；`z = y`（gpvdm 约定）。场分布查询与作图横轴均为 Oghma `y`。

**运行前提**：在 `simulation_core` 根目录执行 `source scripts/init-simulation-build-env.sh build`，再 `./assets/ipynb/simulation/TMM/run_tmm.sh jupyter`（或在已 source 的环境中打开本 notebook）。须具备 `SIMULATION_ARTIFACTS_DIR`（Release `build/`）与 `SIMULATION_DATABASE_DIR`（YAML `assets/database`）；**勿**使用 `init-toykits-build-env.sh` / `.simulation_toolkits`。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from oghma_runtime import bootstrap_tmm_session, oghma_project_dir

_, RUNTIME, TMM_DIR = bootstrap_tmm_session()
import simulation

from oghma_core import (
    compare_metrics,
    epitaxy_table,
    format_oghma_epitaxy_roles,
    load_oghma_optical_reference,
    load_oghma_project,
    normalize_rows,
import simulation_database_parser as sdp
)
from oghma_oled_utils import (
    compute_oled_emission_stack_rt_ito_al,
    compute_oled_passive_rt_ito_al,
    diagnose_oled_stack,
    format_oghma_halfspaces_ito_al,
    ito_contact_thickness_um,
    list_oghma_optical_snapshots,
    load_oghma_optical_snapshot,
    OLED_RT_GATES,
)
from oghma_pytest_helpers import rt_gate_passed

OGHMA_PROJECT_DIR = oghma_project_dir("oled", "01_hello_oled")
PROJECT = load_oghma_project(OGHMA_PROJECT_DIR)
REF = load_oghma_optical_reference(PROJECT)
wl_um = REF["reflect_wl_um"]
R_BASE = REF["reflect"]
T_BASE = REF["transmit"]
y_oghma_um = REF["photons_y_um"]
ALIGNMENT_REPORT = []
RT_GATE_PASSED = False
STACK_CONFIG = {}
ITO_T = ito_contact_thickness_um(PROJECT)

print(f"RUNTIME={RUNTIME}")
print(f"materials: {sdp.materials_root(init=True)}")
print(f"simmode: {PROJECT.simmode}, thickness: {PROJECT.total_thickness_um * 1e3:.1f} nm")
print(f"λ: {wl_um[0]:.3f}–{wl_um[-1]:.3f} μm ({len(wl_um)} pts)")


from oghma_fdtd_alignment import report_oled_metrics


def report_metrics(name, tmm, baseline, *, x=None, rmse_thr=0.02, max_thr=0.05, peak_thr=None, min_corr=None):
    return report_oled_metrics(
        ALIGNMENT_REPORT, name, tmm, baseline,
        x=x, rmse_thr=rmse_thr, max_thr=max_thr, peak_thr=peak_thr, min_corr=min_corr,
    )


def plot_1d_compare(x, tmm, baseline, ylabel, title, baseline_name):
    err = tmm - baseline
    fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    axes[0].plot(x, tmm, label="TMM")
    axes[0].plot(x, baseline, "--", label=f"{baseline_name} (baseline)")
    axes[0].set_ylabel(ylabel)
    axes[0].set_title(title)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(x, err, color="C3")
    axes[1].axhline(0, color="k", lw=0.5)
    axes[1].set_xlabel("λ (μm)" if "λ" in title or "spectrum" in title.lower() else "y (μm)")
    axes[1].set_ylabel(f"Δ{ylabel}")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_2d_compare(wl, y, tmm_grid, baseline_grid, title):
    tg = normalize_rows(tmm_grid.copy())
    bg = normalize_rows(baseline_grid.copy())
    err = np.abs(tg - bg)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, data, lab in zip(axes[:2], [tg, bg], ["TMM", "baseline"]):
        im = ax.pcolormesh(wl, y, data.T, shading="auto", cmap="viridis")
        ax.set_xlabel("λ (μm)")
        ax.set_ylabel("y (μm)")
        ax.set_title(lab)
        plt.colorbar(im, ax=ax, label="row-norm")
    im3 = axes[2].pcolormesh(wl, y, err.T, shading="auto", cmap="magma")
    axes[2].set_xlabel("λ (μm)")
    axes[2].set_ylabel("y (μm)")
    axes[2].set_title("|error|")
    plt.colorbar(im3, ax=axes[2], label="|error|")
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()
    return tg, bg, err


In [2]:
import numpy as np
from oghma_core import _layers_from_builder, _spectrum_y_map
from oghma_oled_utils import (
    build_oghma_oled_passive_stack_ito_al,
    compute_oled_passive_field_intensity_profile_ito_al,
    compute_oled_passive_absorption_profile_ito_al,
)

def verify_rt_conservation(
    r: np.ndarray,
    t: np.ndarray,
    *,
    a: np.ndarray | None = None,
) -> dict[str, float]:
    """Check R+T(+A)≈1 for passive stacks; report mean deviation."""
    r_arr = np.asarray(r, dtype=float)
    t_arr = np.asarray(t, dtype=float)
    total = r_arr + t_arr
    if a is not None:
        total = total + np.asarray(a, dtype=float)
    residual = total - 1.0
    return {
        "mean_r": float(np.mean(r_arr)),
        "mean_t": float(np.mean(t_arr)),
        "mean_total": float(np.mean(total)),
        "max_abs_residual": float(np.max(np.abs(residual))),
    }



def compute_oled_passive_RT_ito_al(project, wl_um, simulation_module, *, incident_angle_rad=0.0):
    def build_layers(wl):
        return build_oghma_oled_passive_stack_ito_al(project, wl, simulation_module)
    wl_arr = np.atleast_1d(np.asarray(wl_um, dtype=float))
    layers = _layers_from_builder(build_layers, float(wl_arr[0]))
    r_list, t_list = simulation_module.TMM_solver_spectrum_rt_power_unpolarized_s(
        layers, wl_arr.tolist(), complex(float(incident_angle_rad), 0.0)
    )
    return np.asarray(r_list, dtype=float), np.asarray(t_list, dtype=float)




def compute_oled_passive_field_intensity_map_ito_al(project, wl_um, y_um, simulation_module, **kwargs):
    return _spectrum_y_map(wl_um, y_um, lambda wl, y: compute_oled_passive_field_intensity_profile_ito_al(project, wl, y, simulation_module, **kwargs))




def compute_oled_passive_absorption_map_ito_al(project, wl_um, y_um, simulation_module, **kwargs):
    return _spectrum_y_map(wl_um, y_um, lambda wl, y: compute_oled_passive_absorption_profile_ito_al(project, wl, y, simulation_module, **kwargs))






## §1 器件堆栈

Oghma epitaxy（底→顶）与 TMM 被动堆栈（半无限 ITO bookend + epitaxy + 半无限 Al bookend）。TMM `z` 为深度；Oghma `y = z`。


In [ ]:
print(format_oghma_epitaxy_roles(PROJECT))
print(format_oghma_halfspaces_ito_al(PROJECT))
display(pd.DataFrame(epitaxy_table(PROJECT)))

diag = diagnose_oled_stack(
    PROJECT, 0.520, simulation,
    stack_builder=build_oghma_oled_passive_stack_ito_al,
)
print(f"TMM passive ITO/Al stack @ 520 nm (ITO_T={ITO_T * 1e3:.0f} nm):")
diag_df = pd.DataFrame(diag)
diag_df["y_oghma_nm"] = diag_df["z_start_um"] * 1e3
display(diag_df)

fig, ax = plt.subplots(figsize=(10, 2.5))
stack_rows = list(reversed(diag))
labels = [row["label"] for row in stack_rows]
depths = [row["depth_um"] * 1e3 for row in stack_rows]
ax.barh(range(len(depths)), depths, color="steelblue")
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.set_xlabel("thickness (nm)")
ax.set_title("Layer thickness (ITO/Al passive stack)")
plt.tight_layout()
plt.show()


## §2 Case — R/T 对齐（Phase 1 门控）

被动 y0 入射，**ITO/Al 半平面**堆栈（`TMM_get_r_t_power_s` / `TMM_get_r_t_power_p`），与 Oghma `optical_output` R/T 对比。


In [ ]:
STACK_CONFIG = {"api": "passive_ito_al", "label": "passive ITO/Al (y0)"}
print("STACK_CONFIG:", STACK_CONFIG)

R_TMM, T_TMM = compute_oled_passive_RT_ito_al(
    PROJECT, wl_um, simulation
)
cons = verify_rt_conservation(R_TMM, T_TMM)
print("R+T conservation (passive ITO/Al):", cons)


In [ ]:
# R: max|Δ|≈0.088 at λ≈303 nm (UV tabulated extrapolation); allow 0.09 for single-point boundary
stats_R = report_metrics("R(λ)", R_TMM, R_BASE, x=wl_um, rmse_thr=0.035, max_thr=0.09, min_corr=0.99)
plot_1d_compare(wl_um, R_TMM, R_BASE, "R", "Reflection spectrum", "R")

# T: ITO/Al 有限 Al 垫层 vs Oghma ABC(y1) → 量级偏低，corr 不可靠；以 RMSE/max 为准
stats_T = report_metrics("T(λ)", T_TMM, T_BASE, x=wl_um, rmse_thr=0.02, max_thr=0.05)
plot_1d_compare(wl_um, T_TMM, T_BASE, "T", "Transmission spectrum", "T")

RT_GATE_PASSED = (
    stats_R["rmse"] < 0.035 and stats_R["max_abs"] < 0.09 and stats_R["corr"] > 0.99
    and stats_T["rmse"] < 0.02 and stats_T["max_abs"] < 0.05
)
print(f"RT_GATE_PASSED = {RT_GATE_PASSED}")
if not RT_GATE_PASSED:
    i_r = int(np.argmax(np.abs(R_TMM - R_BASE)))
    print(f"Worst R @ λ={wl_um[i_r]:.3f} μm: TMM={R_TMM[i_r]:.4f}, baseline={R_BASE[i_r]:.4f}")
    print(f"T mean: TMM={T_TMM.mean():.3e}, baseline={T_BASE.mean():.3e}, ratio={T_TMM.mean()/T_BASE.mean():.3f}")


### 分析 §2

- 与 02（glass/air）不同：底/顶半平面为 ITO / Al 垫层；器件仍从 Oghma `y=0` 起。
- R 通常优于 02；T 绝对误差小但量级偏低（有限 Al vs ABC），不以 corr 门控。
- 若 FAIL：检查材料库路径、层厚 (nm)、ITO/Al 垫层厚度是否与 epitaxy contact 一致。


## §3 Case — ⟨R⟩, ⟨T⟩

光谱平均 vs `light_stats.json`。


In [ ]:
ls = REF.get("light_stats", {})
R_mean_base = float(ls.get("R", np.mean(R_BASE)))
T_mean_base = float(ls.get("T", np.mean(T_BASE)))
scalars_tmm = np.array([R_TMM.mean(), T_TMM.mean()])
scalars_base = np.array([R_mean_base, T_mean_base])
report_metrics("⟨R⟩,⟨T⟩", scalars_tmm, scalars_base, rmse_thr=0.02, max_thr=0.02)
print(f"baseline light_stats: R={R_mean_base:.4f}, T={T_mean_base:.2e}")


## §4 Phase 1 门控

R/T 未 PASS 时禁止 Phase 2。


In [ ]:
assert RT_GATE_PASSED, (
    "R/T 未对齐 — 禁止 Phase 2。回到 §2 检查 STACK_CONFIG、材料库与边界。"
)
print("Phase 1 gate PASSED — proceeding to Phase 2")


## §5 Case — photons(λ,y)

被动 y0 |E|² vs `photons_yl.csv`（行归一化比形态）。


In [ ]:
PHOTONS_TMM = compute_oled_passive_field_intensity_map_ito_al(
    PROJECT, wl_um, y_oghma_um, simulation,
)
PHOTONS_BASE = REF["photons_grid"]
PHOTONS_TMM_N, PHOTONS_BASE_N, _ = plot_2d_compare(
    wl_um, y_oghma_um, PHOTONS_TMM, PHOTONS_BASE, "Photon density (row-normalized, ITO/Al)"
)
report_metrics("photons(λ,y) norm", PHOTONS_TMM_N, PHOTONS_BASE_N, rmse_thr=0.20, max_thr=0.5, min_corr=0.80)

wl_cross = 0.436
i_lam = int(np.argmin(np.abs(wl_um - wl_cross)))
report_metrics(
    f"photons @ {wl_cross * 1e3:.0f} nm",
    PHOTONS_TMM_N[i_lam], PHOTONS_BASE_N[i_lam],
    x=y_oghma_um, rmse_thr=0.10, max_thr=0.30, min_corr=0.90,
)
plot_1d_compare(y_oghma_um, PHOTONS_TMM_N[i_lam], PHOTONS_BASE_N[i_lam],
                "photon density", f"Cross-section @ {wl_cross * 1e3:.0f} nm", "photon density")


## §5b Case — photons_abs(λ,y)

被动吸收 vs `photons_abs_yl.csv`。


In [ ]:
ABS_TMM = compute_oled_passive_absorption_map_ito_al(
    PROJECT, wl_um, y_oghma_um, simulation,
)
ABS_BASE = REF["photons_abs_grid"]
ABS_TMM_N = normalize_rows(ABS_TMM)
ABS_BASE_N = normalize_rows(ABS_BASE)
report_metrics("photons_abs(λ,y) norm", ABS_TMM_N, ABS_BASE_N, rmse_thr=0.05, max_thr=0.2, min_corr=0.99)
_=plot_2d_compare(wl_um, y_oghma_um, ABS_TMM_N, ABS_BASE_N, "Absorption (row-normalized, ITO/Al)")


## §6 Case — 光学快照

`optical_snapshots/*/photons.csv` vs 被动 |E|² 截面。


In [ ]:
snap_list = list_oghma_optical_snapshots(PROJECT.project_dir)
print(f"{len(snap_list)} snapshots")
for target_wl in (310.0, 436.0, 577.0):
    entry = min(snap_list, key=lambda e: abs((e.get("wl_um") or 0) - target_wl))
    snap = load_oghma_optical_snapshot(PROJECT.project_dir, entry["index"])
    wl = snap["wl_um"]
    y_snap = snap["y_um"]
    prof_tmm = compute_oled_passive_field_intensity_map_ito_al(
        PROJECT, np.array([wl]), y_snap, simulation,
    )[0]
    prof_tmm_n = prof_tmm / max(prof_tmm.max(), 1e-30)
    prof_base_n = snap["photons"] / max(snap["photons"].max(), 1e-30)
    report_metrics(
        f"snapshot @ {wl:.0f} nm", prof_tmm_n, prof_base_n,
        x=y_snap, rmse_thr=0.25, max_thr=0.35, min_corr=0.75,
    )
    plot_1d_compare(y_snap, prof_tmm_n, prof_base_n, "photon density",
                    f"Snapshot @ {wl:.0f} nm", "photon density")


## §7 汇总对齐报告


In [ ]:
summary = pd.DataFrame(ALIGNMENT_REPORT)
display(summary)
n_pass = int((summary["status"] == "PASS").sum())
print(f"PASS: {n_pass}/{len(summary)}")
print(f"Locked stack: {STACK_CONFIG.get('label', 'n/a')}")
print(f"ITO contact 0–150 nm @ z≥0 (Oghma y=0); semi-inf bookends: ITO(∞) @ z=−∞, Al(∞) @ z=+∞")
